### Extraction of Data for plotting of MP2-cylinder-paper data.
- create index table which contains all simulations with its parameters, stats and the status if drag and lift timeseries are available
- extract chosen data timeseries for drag and lift coefficient and save them in a new folder with all relevant parameters in their name
- FUTURE: do statistics on data, pre-analyze (plot), incorporate data in plots of cylinder-paper

#### IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from pathlib import Path
import time
import matplotlib.pyplot as plt

### INITIALIZATION of FUNCTIONS and FUNCTIONALITY:

In [ ]:
# ==========================================
# 1. HELFER-FUNKTIONEN
# ==========================================

def get_time_range(file_path):
    """Liest Start und Ende der Zeitreihe aus (überspringt Header)."""
    try:
        with open(file_path, 'rb') as f:
            f.readline()  # Header überspringen
            first_line = f.readline().decode().split()
            if not first_line: return None, None
            # Wir nehmen an, timePU ist das zweite Element (Index 1)
            t_start = float(first_line[1])

            f.seek(0, os.SEEK_END)
            pointer = f.tell() - 2
            while pointer > 0:
                f.seek(pointer)
                if f.read(1) == b'\n': break
                pointer -= 1
            last_line = f.readline().decode().split()
            t_end = float(last_line[1]) if len(last_line) > 1 else t_start
            return t_start, t_end
    except:
        return None, None

def parse_stats_file(file_path):
    """Extrahiert Parameter aus der *_parms_stats_obs.txt."""
    content = file_path.read_text()
    def search(pattern, dtype=float):
        match = re.search(pattern, content, re.IGNORECASE)
        if match:
            val = match.group(1).strip()
            return dtype(val) if dtype else val
        return None

    return {
        'Re': search(r'Re\s*=\s*([\d\.]+)'),
        'Ma': search(r'Ma\s*=\s*([\d\.]+)'),
        'n_steps': search(r'n_steps\s*=\s*(\d+)', int),
        'T_target': search(r'T_target\s*=\s*([\d\.]+)'),
        'GPD': search(r'gridpoints_per_diameter \(gpd\)\s*=\s*(\d+)', int),
        'DpX': search(r'DpX \(D/X\)\s*=\s*([\d\.]+)'),
        'DpY': search(r'DpY \(D/Y\)\s*=\s*([\d\.]+)'),
        'DpZ': search(r'DpZ \(D/Z\)\s*=\s*([\d\.]+)'),
        'bc_type': search(r'bc_type\s*=\s*(\w+)', str),
        'stencil': search(r'stencil\s*=\s*(\w+)', str),
        'collision': search(r'collision\s*=\s*(\w+)', str),
    }

# ==========================================
# 2. HAUPT-FUNKTIONEN
# ==========================================

def build_master_index(base_dir):
    """Scannt alle Ordner und erstellt Roh-Index-Datenbank"""
    base_path = Path(base_dir)
    records = []
    stats_files = list(base_path.rglob("*_parms_stats_obs.txt"))
    print(f"Scanne {len(stats_files)} Ordner...")

    for sf in stats_files:
        folder = sf.parent
        data = parse_stats_file(sf)
        data['Folder'] = folder.name
        data['path'] = str(folder)

        drag_files = list(folder.glob("drag*.txt"))
        if drag_files:
            data['time_start'], data['time_end'] = get_time_range(drag_files[0])
            data['has_drag'] = True
            data['has_lift'] = any(folder.glob("lift*.txt"))
        else:
            data['time_start'] = data['time_end'] = None
            data['has_drag'] = data['has_lift'] = False
        records.append(data)

    return pd.DataFrame(records)

In [ ]:
def analyze_simulation_continuity(df):
    """Gruppiert Simulationen und prüft auf Lücken."""
    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'bc_type', 'stencil', 'collision']
    df_sorted = df.sort_values(by=core_params + ['time_start']).copy()

    analysis = []
    for name, group in df_sorted.groupby(core_params, dropna=False):
        group = group.reset_index(drop=True)
        for i in range(len(group)):
            row = group.iloc[i].to_dict()
            issues = []
            if i == 0 and row['time_start'] and row['time_start'] > 0.5:
                issues.append(f"Start-Lücke ({row['time_start']}s)")
            if i > 0:
                gap = row['time_start'] - group.iloc[i-1]['time_end']
                if gap > 0.1: issues.append(f"Lücke: {gap:.2f}s")
                if gap < -0.1: issues.append(f"Überlapp: {abs(gap):.2f}s")

            row['Status'] = " / ".join(issues) if issues else "OK"
            analysis.append(row)
    return pd.DataFrame(analysis)

In [ ]:
def merge_simulation_chains(df_checked, output_base_dir="./merged_results"):
    """verkettet alle Tiole einer Simulation mit identischen Parametern (außer Zeit) zu einer einzelnen Zeitreihe. Dabei werden überlappende Regionen entfernt (bzw. nur einmal berücksichtig)"""
    output_path = Path(output_base_dir)
    output_path.mkdir(exist_ok=True, parents=True)

    # Gruppierungsparameter (ohne DpZ für 2D-Kompatibilität)
    core_params = ['Re', 'Ma', 'GPD', 'DpY', 'stencil', 'bc_type', 'collision']
    df_valid = df_checked[df_checked['has_drag'] == True].copy()

    groups = list(df_valid.groupby(core_params, dropna=False))
    print(f"Starte Merging von {len(groups)} Gruppen...")

    for name, group in groups:
        # Chronologisch sortieren
        group = group.sort_values('time_start')
        first_row, last_row = group.iloc[0], group.iloc[-1]

        # Dateiname bauen
        t_end_round = int(round(last_row['time_end'])) if last_row['time_end'] else 0
        name_parts = [
            f"Re{first_row['Re']}", f"Ma{first_row['Ma']}",
            f"GPD{int(first_row['GPD'] or 0)}", f"{first_row['stencil']}",
            f"DpY{first_row['DpY']}", f"BC_{first_row['bc_type']}",
            f"Coll_{first_row['collision']}"
        ]
        if "D3" in str(first_row['stencil']): name_parts.append(f"DpZ{first_row['DpZ']}")
        name_parts.append(f"T{t_end_round}")
        base_filename = "_".join(filter(None, [str(p) for p in name_parts]))

        for mode in ['drag', 'lift']:
            combined_chunks = []
            target_col = 'Cd' if mode == 'drag' else 'Cl'

            for _, row in group.iterrows():
                folder = Path(row['path'])
                files = list(folder.glob(f"{mode}*.txt"))
                if not files: continue

                try:
                    # DER SCHNELLE PARSER:
                    # Wir ignorieren den Header (Zeilen mit #) komplett.
                    # Wir wissen: Es sind genau 3 Spalten mit Leerzeichen getrennt.
                    df_chunk = pd.read_csv(
                        files[0],
                        sep=r'\s+',
                        engine='c',    # C-Engine für maximale Geschwindigkeit
                        comment='#',   # Ignoriert die Header-Zeile
                        header=None,   # Wir definieren die Namen selbst
                        names=['stepLU', 'timePU', target_col]
                    )

                    if not df_chunk.empty:
                        combined_chunks.append(df_chunk)
                except Exception as e:
                    print(f"Fehler in {files[0].name}: {e}")

            if not combined_chunks: continue

            # Alle Fragmente zusammenkleben
            full_df = pd.concat(combined_chunks, ignore_index=True)

            # Duplikate an den Nahtstellen entfernen (5 Nachkommastellen Toleranz)
            full_df['time_rounded'] = full_df['timePU'].round(5)
            full_df = full_df.drop_duplicates(subset=['time_rounded'], keep='first')
            full_df = full_df.drop(columns=['time_rounded']).sort_values('timePU')

            # Speichern als saubere CSV
            target_file = output_path / f"{base_filename}_{mode}.csv"
            full_df.to_csv(target_file, index=False)
            print(f"Datei fertig: {target_file.name} ({len(full_df)} Zeilen)")

# Ausführung
# merge_simulation_chains(df_checked, "./dein_output_pfad")

In [ ]:
# CHECK Unstetigkeit in den Zeitreihen
def scan_all_merges_for_jumps(merged_dir, rel_threshold=0.05, abs_threshold=0.005):
    """
    Scannt alle CSVs im Ordner auf Sprünge an den Nahtstellen.
    rel_threshold: 0.05 entspricht 5% Abweichung zum Vorwert.
    abs_threshold: Mindest-Absolutwert (um Rauschen bei Werten nahe 0 zu ignorieren).
    """
    merged_path = Path(merged_dir)
    csv_files = list(merged_path.glob("*.csv"))
    report = []

    print(f"Scanne {len(csv_files)} Dateien auf Unstetigkeiten...")

    for file in csv_files:
        df = pd.read_csv(file)
        val_col = 'Cd' if 'drag' in file.name else 'Cl'

        # Nahtstellen finden (wo stepLU sinkt oder gleich bleibt)
        transition_indices = df.index[df['stepLU'].diff() <= 0].tolist()

        for idx in transition_indices:
            # Wir brauchen den Wert direkt davor (idx-1) und den an der Naht (idx)
            val_before = df.loc[idx-1, val_col]
            val_after = df.loc[idx, val_col]

            diff = abs(val_after - val_before)

            # Relative Abweichung berechnen (Vorsicht bei Division durch ~0)
            denom = abs(val_before) if abs(val_before) > 1e-6 else 1e-6
            rel_diff = diff / denom

            # Nur melden, wenn sowohl relativer als auch absoluter Sprung relevant sind
            if rel_diff > rel_threshold and diff > abs_threshold:
                error_data = {
                    'file': file.name,
                    'time': df.loc[idx, 'timePU'],
                    'val_before': val_before,
                    'val_after': val_after,
                    'jump_pct': rel_diff * 100
                }
                report.append(error_data)
                print(f"⚠️ Sprung in {file.name}: {rel_diff:.1%} bei t={df.loc[idx, 'timePU']}")

    return pd.DataFrame(report)

# --- AUTOMATISCHE PLOT-FUNKTION FÜR PROBLEMFAELLE ---
def plot_problem_cases(report_df, merged_dir, window=100):
    """Erstellt Plots nur für die Dateien, in denen Sprünge gefunden wurden."""
    for _, row in report_df.iterrows():
        file_path = Path(merged_dir) / row['file']
        df = pd.read_csv(file_path)
        val_col = 'Cd' if 'drag' in row['file'] else 'Cl'

        # Finde den Index für die Zeit im Report
        idx = df.index[df['timePU'] == row['time']][0]

        start, end = max(0, idx-window), min(len(df), idx+window)
        subset = df.iloc[start:end]

        plt.figure(figsize=(8, 4))
        plt.plot(subset['timePU'], subset[val_col], 'b-o', markersize=2, label='Daten')
        plt.axvline(row['time'], color='red', linestyle='--', label='Nahtstelle')
        plt.title(f"Sprung-Detail: {row['file']}\nZeit: {row['time']:.2f}")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()

# Ausführung:
# df_jumps = scan_all_merges_for_jumps("./merged_results")
# if not df_jumps.empty:
#     plot_problem_cases(df_jumps, "./merged_results")

#### important index-loading or -creation functionality
Lade oder erstelle Haupt-Master-Index

In [ ]:
# (!) Dateipfade für die Index-Zwischenspeicher
FILE_RAW_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_raw.csv"
FILE_CHECKED_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_checked.csv"

In [ ]:
# ZWISCHENSPEICHERPUNKTE: raw-index und checked-index

#siehe oben: (!) Dateipfade für die Index-Zwischenspeicher
#siehe oben: FILE_RAW_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_raw.csv"
#siehe oben: FILE_CHECKED_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_checked.csv"

def get_or_create_index(data_input_dir, force_rescan=False):
    """HAUPT-STEUER-FUNKTION: Lädt oder erstelle Haupt-Index-File"""
    # 1. Prüfen, ob der analysierte Index schon da ist
    if Path(FILE_CHECKED_INDEX).exists() and not force_rescan:
        print(f"Lade existierenden Analyse-Index aus {FILE_CHECKED_INDEX}...")
        return pd.read_csv(FILE_CHECKED_INDEX)

    # 2. Falls nicht, prüfen ob zumindest der rohe Scan da ist
    if Path(FILE_RAW_INDEX).exists() and not force_rescan:
        print(f"Lade rohen Scan aus {FILE_RAW_INDEX} und starte Analyse...")
        df_raw = pd.read_csv(FILE_RAW_INDEX)
    else:
        # 3. Komplett neu scannen (dauert am längsten)
        print("Scanne Verzeichnisse neu (das kann dauern)...")
        df_raw = build_master_index(data_input_dir)
        df_raw.to_csv(FILE_RAW_INDEX, index=False)

    # Analyse durchführen
    print("Führe Kontinuitätsanalyse durch...")
    df_checked = analyze_simulation_continuity(df_raw)
    df_checked.to_csv(FILE_CHECKED_INDEX, index=False)

    return df_checked

In [ ]:
# Summary des df_Checked OHNE merge
def create_final_summary(df_checked):
    # Definition der Spalten, die in die Übersicht sollen
    summary_cols = [
        'Re', 'Ma', 'n_steps', 'T_target', 'GPD',
        'DpX', 'DpY', 'DpZ', 'bc_type', 'stencil', 'collision'
    ]

    # Wir gruppieren nach diesen Parametern.
    # .first() nimmt die Werte der ersten Simulation in der Kette (die sind identisch)
    # .max() bei time_end liefert uns die tatsächlich erreichte Zeit der gesamten Kette
    summary_df = df_checked.groupby(summary_cols, dropna=False).agg({
        'time_end': 'max',
        'path': 'count' # Hilfreich um zu sehen, aus wie vielen Fragmenten die Kette bestand
    }).reset_index()

    # Spalten umbenennen für mehr Klarheit
    summary_df = summary_df.rename(columns={'path': 'num_fragments'})

    # Sortieren für bessere Lesbarkeit (z.B. nach Reynolds-Zahl)
    summary_df = summary_df.sort_values(['Re', 'Ma', 'GPD'])

    return summary_df

In [ ]:
# SUMMARY-DF aus den tatsächlichen ge-merge-ten Zeitreihen
def create_summary_from_merged_files(merged_dir):
    """Erstellt Übersicht über die verketteten Zeitreihen auf Basis der tatsächlich auf Festplatte vorhandenen Daten"""
    merged_path = Path(merged_dir)
    # Wir suchen nur nach den drag-Dateien (da lift meist identisch ist)
    files = list(merged_path.glob("*_drag.csv"))

    summary_data = []

    print(f"Analysiere {len(files)} Master-Dateien...")

    for f in files:
        # 1. Metadaten aus dem Dateinamen extrahieren
        # Beispiel: Re100.0_Ma0.05_GPD50_D2Q9_DpY50.0_BC_hwbbc2_Coll_bgk_T300_drag.csv
        name = f.stem  # Dateiname ohne .csv

        def extract(pattern, string):
            match = re.search(pattern, string)
            return match.group(1) if match else None

        # 2. Die Datei kurz einlesen, um echte Werte zu kriegen
        # Wir lesen nur die erste und letzte Zeile für Speed
        try:
            # Wir nutzen dask oder nur skiprows/nrows um nicht
            # die ganze Riesen-Datei zu laden
            first_row = pd.read_csv(f, nrows=1)
            # Für die letzte Zeile:
            df_full = pd.read_csv(f) # Falls Dateien nicht Giga-Byte groß sind

            last_time = df_full['timePU'].max()
            row_count = len(df_full)

            summary_data.append({
                'Re': float(extract(r'Re([\d\.]+)', name) or 0),
                'Ma': float(extract(r'Ma([\d\.]+)', name) or 0),
                'GPD': int(extract(r'GPD(\d+)', name) or 0),
                'stencil': extract(r'_(D\dQ\d+)_', name),
                'DpY': float(extract(r'DpY([\d\.]+)', name) or 0),
                'DpZ': float(extract(r'DpZ([\d\.]+)', name) or 0) if "D3" in name else None,
                'bc_type': extract(r'BC_([^_]+)', name),
                'collision': extract(r'Coll_([^_]+)', name),
                'real_time_end': last_time,
                'total_steps_recorded': row_count,
                'file_name': f.name
            })
        except Exception as e:
            print(f"Fehler bei Datei {f.name}: {e}")

    df_final = pd.DataFrame(summary_data)

    # Sortieren für die Übersicht
    df_final = df_final.sort_values(['Re', 'Ma', 'GPD'])

    return df_final

# --- ANWENDUNG ---
# final_master_summary = create_summary_from_merged_files("./merged_data")
# final_master_summary.to_csv("final_production_overview.csv", index=False)

In [ ]:
def load_simulation(summary_df, merged_dir, Re, Ma, GPD):
    """Sucht die passende Datei aus der Summary und lädt sie."""
    match = summary_df[
        (summary_df['Re'] == Re) &
        (summary_df['Ma'] == Ma) &
        (summary_df['GPD'] == GPD)
    ]

    if not match.empty:
        file_path = Path(merged_dir) / match.iloc[0]['file_name']
        return pd.read_csv(file_path)
    else:
        print("Keine passende Simulation gefunden.")
        return None

### MAIN USER PROGRAM:

#### Define Paths and Index-File-Names

In [ ]:
# Pfade
INPUT_DIR = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/00_CP-data_raw"
OUTPUT_MERGED = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/01_CP-data_processed"

#Index Files:
#   - siehe oben über get_or_create_index()
FILE_RAW_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_raw.csv"
FILE_CHECKED_INDEX = "/mnt/ScratchHDD2/MBille/01_LBM_CP-data/master_index_checked.csv"

#### (optional) Merge for segmented Time Series

In [ ]:
# Das Mergen musst du nur einmal ausführen. Danach liegen die fertigen
# ...Zeitreihen ja bereits in OUTPUT_MERGED.
if False:
    merge_simulation_chains(df_checked, OUTPUT_MERGED)

#### Load or create Index

In [ ]:
# Holt den Index (entweder von Festplatte oder durch neuen Scan)
df_checked = get_or_create_index(INPUT_DIR, force_rescan=False)

#### (optional) Get Summary-Overview WITHOUT Merge

In [ ]:
# Erstelle Summary OHNE Merge
if False:
    # SUMMARY ohne MERGE
    final_summary = create_final_summary(df_checked)

    # Speichern als CSV für Dokumentation oder Excel
    final_summary.to_csv("/mnt/ScratchHDD2/MBille/01_LBM_CP-data/"+ "master_overview_non-merged.csv", index=False)
    # Anzeige der ersten Zeilen
    print("Master-Übersicht erstellt:")
    print(final_summary.head())

#### Create and save Summary-Overview over all merged files

In [ ]:
# Erstelle Summary MIT Merge (basis für weitere Auswertung
final_master_summary = create_summary_from_merged_files(OUTPUT_MERGED)
final_master_summary.to_csv("master_overview_merged.csv", index=False)

### STATISTICS